<div style="text-align: center;">
  <span style="
      color: lightskyblue;
      font-family: 'Segoe UI', 'Noto Sans KR', 'Malgun Gothic', sans-serif;
      font-weight: 700;
      font-size: 48px;
      letter-spacing: 0.5px;
      line-height: 1.2;
  ">
    Demo File for Pick and Place
  </span>
</div>

<span style="
    color:#B8860B;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:50px;
    letter-spacing:1px;
    line-height:1.2;
">
Import Library
</span>

In [1]:
import rby1_sdk as rby
import numpy as np
import rby1_lib_ysw as ysw
import logging
import sys
import time
import pyrealsense2 as rs
import cv2

np.set_printoptions(suppress=True, precision=4, linewidth=300)  

# In .ipynb, show logging 
logging.basicConfig(
    level=logging.INFO,
    format="[%(levelname)s] %(message)s",
    # format="%(asctime)s [%(levelname)s] %(message)s",
    stream=sys.stdout,
    force=True,
)

<span style="
    color:#B8860B;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:50px;
    letter-spacing:1px;
    line-height:1.2;
">
Initialization [Connect + PowerON]
</span>

In [3]:
# robotConfig = ysw.setRobotConfig(ysw.MODE_SIM, ysw.MODEL_A)                 # For Simulation
# robotConfig = ysw.setRobotConfig(ysw.MODE_LOCAL_COMPUTER, ysw.MODEL_A)      # For Real Robot + Laptop (RBY1-Hotspot)
robotConfig = ysw.setRobotConfig(ysw.MODE_DIRECT_UPC, ysw.MODEL_A)          # For Real Robot + UPC

robot = ysw.initialize(robotConfig.mode, robotConfig.address, robotConfig.model, option=ysw.CONNECT_AND_POWER)
# robot = ysw.initialize(robotConfig.mode, robotConfig.address, robotConfig.model, option=ysw.CONNECT_ONLY)
if (robot != None):
    model = robot.model()

*************************** [Initialization Starts] ****************************

[INFO] SUCCESS: Connected to the Robot!
[INFO] SUCCESS: Power ON the Robot!
[INFO] SUCCESS: Servo ON the Robot!
[INFO] SUCCESS: Reset the Control Manager!
[INFO] SUCCESS: Enable the Control Manager!
[INFO] SUCCESS: Enable the Grippers!

*************************** [Intialization is Done] ****************************





<span style="
    color:#B8860B;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:50px;
    letter-spacing:1px;
    line-height:1.2;
">
Power OFF
</span>

In [15]:
ysw.powerOFF(robot, robotConfig)

****************************** [powerOFF Starts] *******************************

[INFO] SUCCESS: Disable Grippers!
[INFO] SUCCESS: Disable the Control Manager!

****************************** [PowerOFF is Done] ******************************





True

<span style="
    color:#B8860B;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:50px;
    letter-spacing:1px;
    line-height:1.2;
">
Posture [Initial]
</span>

In [4]:
desired_pos = ysw.DEMO_BASIC_POS_260723     # ZERO_POS, READY_POS, PACK_POS, DEMO_BASIC_POS_260723
ysw.positionCtrl_joint_22dof(robot, desired_pos, minimum_time=5.0, control_hold_time=0.5)

******************** [22-DoF Joint Position Control Starts] ********************


********************* [22-DoF Joint Position Control Ends] *********************





True

In [5]:
desired_pos = ysw.DEMO_VISION_BASIC_POS_260723     # ZERO_POS, READY_POS, PACK_POS, DEMO_BASIC_POS_260723
ysw.positionCtrl_joint_22dof(robot, desired_pos, minimum_time=5.0, control_hold_time=0.5)

******************** [22-DoF Joint Position Control Starts] ********************


********************* [22-DoF Joint Position Control Ends] *********************





True

<span style="
    color:#B8860B;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:50px;
    letter-spacing:1px;
    line-height:1.2;
">
Posture [Turn to Shelf]
</span>

In [6]:
desired_pos = ysw.DEMO_TURN_POS_260723     # ZERO_POS, READY_POS, PACK_POS, DEMO_TURN_POS_260723
ysw.positionCtrl_joint_22dof(robot, desired_pos, minimum_time=5.0, control_hold_time=0.5)

******************** [22-DoF Joint Position Control Starts] ********************


********************* [22-DoF Joint Position Control Ends] *********************





True

<span style="
    color:#B8860B;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:50px;
    letter-spacing:1px;
    line-height:1.2;
">
Leftmost Object Detection during 4 seconds
</span>

In [6]:
d435i = ysw.D435i(rgb_on=True, depth_on=True, pixel=(640, 480), fps=30)
d435i.initialize()

record_duration = 4.0  # [sec]

leftmost_object_samples = []        # 제일 왼쪽 물체

d435i.startPipeline()

try:
    start_time = time.time()

    while True:
        elapsed_time = time.time() - start_time

        if elapsed_time >= record_duration:
            break

        color_frame, depth_frame = d435i.getRawFrames()

        if color_frame is None or depth_frame is None:
            continue

        color_image, depth_image = d435i.frameToImage(color_frame, depth_frame)

        detected_objects, color_image, mask = d435i.detectColorObjectContour3D(
            color_image=color_image,
            depth_frame=depth_frame,
            mask_list=ysw.HSV_MASK_BLACK,
            minimum_area=1000,       
            maximum_depth=0.8,                   # ignore distant object larger than x[m] 
        )

        if len(detected_objects) == 0:
            continue

        # 카메라 기준 X 좌표가 작을수록 이미지 왼쪽
        # RealSense/D435i 기준: X+ = image right
        obj_list_from_left = sorted(
            detected_objects,
            key=lambda obj: obj["center_3d"][0],
            reverse=False
        )

        # 제일 왼쪽 물체 하나만 선택
        obj = obj_list_from_left[0]

        # 제일 왼쪽 물체 sample만 저장
        leftmost_object_samples.append({
            "area_2d":      float(obj["area_2d"]),
            "pixel_center": np.array(obj["pixel_center"], dtype=np.float64),
            "center_3d":    np.array(obj["center_3d"], dtype=np.float64),
            "bbox_3d_size": np.array(obj["bbox_3d_size"], dtype=np.float64),
            "bbox_3d_min":  np.array(obj["bbox_3d_min"], dtype=np.float64),
            "bbox_3d_max":  np.array(obj["bbox_3d_max"], dtype=np.float64),
        })

finally:
    d435i.stopPipeline()
    d435i.destroyAllWindows()



# =========================
# Average leftmost object
# =========================

leftmost_obj_avg_sample = None

if len(leftmost_object_samples) > 0:
    leftmost_obj_avg_sample = ysw.average_detectColorObjectContour3D(
        leftmost_object_samples
    )

print("\n========== Detection Result ==========")
print("Only the LEFTMOST object is used\n")

if leftmost_obj_avg_sample is None:
    print("No object detected.")

else:
    print("\n[Leftmost Object]")
    print(f"num_samples             : {leftmost_obj_avg_sample['num_samples']}")
    print(f"area_2d_avg             : {leftmost_obj_avg_sample['area_2d_avg']:.1f}")
    print(f"area_2d_median          : {leftmost_obj_avg_sample['area_2d_median']:.1f}")
    print(f"center_3d_avg           : {np.round(leftmost_obj_avg_sample['center_3d_avg'], 4)} m")
    print(f"center_3d_median        : {np.round(leftmost_obj_avg_sample['center_3d_median'], 4)} m")
    print(f"bbox_3d_size_avg        : {np.round(leftmost_obj_avg_sample['bbox_3d_size_avg'], 4)} m")
    print(f"bbox_3d_size_median     : {np.round(leftmost_obj_avg_sample['bbox_3d_size_median'], 4)} m")

[INFO] D435i pipeline started

========== Detection Result ==========
Only the LEFTMOST object is used


[Leftmost Object]
num_samples             : 13
area_2d_avg             : 2417.6
area_2d_median          : 2407.5
center_3d_avg           : [-0.0672 -0.0419  0.695 ] m
center_3d_median        : [-0.0671 -0.0396  0.6945] m
bbox_3d_size_avg        : [0.031  0.1252 0.1376] m
bbox_3d_size_median     : [0.0309 0.1256 0.131 ] m


<span style="
    color:#B8860B;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:50px;
    letter-spacing:1px;
    line-height:1.2;
">
Calculate IK for Leftmost Object
</span>

In [ ]:
# # {A}: vision frame
# # {B}: link_head_2 frame
# # {M}: Goal frame (object) 
# # {M1}: Goal frame (object) before open the gripper 
# # {M2}: Goal frame (object) after open the gripper 
# # {N}: Goal frame (collecting box) 
# # {C}: ee_finger_l1/r1 frame
# # {D}: final fingertip frame -> same orientation of {C}

# leftmost_obj_pos    = leftmost_obj_avg_sample["center_3d_avg"]
# leftmost_obj_3Dsize = leftmost_obj_avg_sample["bbox_3d_size_avg"]

# desiredPos_linkhead2_ee_before_open_gripper, desiredPos_linkhead2_ee_after_open_gripper = ysw.calculateEEpos_forGrasp(leftmost_obj_pos, 
#                                                                                                                       leftmost_obj_3Dsize,
#                                                                                                                       object_offset_use=False, 
#                                                                                                                       manual_offset=0.03)

# moved_arm = None
# if (desiredPos_linkhead2_ee_before_open_gripper[1] <= 0):     
#     moved_arm = "right"
# else:     
#     moved_arm = "left"

# if (moved_arm == "right"):
#     R1 = ysw.rotMat_z(90.0*ysw.DEG2RAD)      # "link_head_2" frame 기준으로 z축 +90도 회전
#     R2 = ysw.rotMat_x(-90.0*ysw.DEG2RAD)     # After R1, converted "link_head_2" frame 기준으로 x축 -90도 회전
#     R_BC_desired = R1 @ R2                   # Make Grasping orientation same as D435i frame orientation angle (it does not mean that x,y,z axis directions are same)  
# elif (moved_arm == "left"):
#     R1 = ysw.rotMat_x(90.0*ysw.DEG2RAD)      # "link_head_2" frame 기준으로 x축 +90도 회전
#     R2 = ysw.rotMat_y(-90.0*ysw.DEG2RAD)     # After R1, converted "link_head_2" frame 기준으로 y축 -90도 회전
#     R_BC_desired = R1 @ R2                   # Make Grasping orientation same as D435i frame orientation angle (it does not mean that x,y,z axis directions are same) 

# # [Theoretical] object_pos_from_linkhead2 = ysw.convertPos_vision_to_linkhead2(object_pos) # P_BM
# P_BM1 = desiredPos_linkhead2_ee_before_open_gripper
# R_BM1 = R_BC_desired
# T_BM1 = ysw.make_HT(R_BM1, P_BM1)    

# P_BM2 = desiredPos_linkhead2_ee_after_open_gripper
# R_BM2 = R_BC_desired
# T_BM2 = ysw.make_HT(R_BM2, P_BM2)    

# # "ee_finger_l1/r1" to final end-effector -> should be physically measured
# pos_from_eeFinger_to_fingertip = np.array([0, 0, ysw.GRIPPER_Z_OFFSET_FROM_LINK_EE_FINGER]) # P_CD
# P_CD = pos_from_eeFinger_to_fingertip
# T_CD = ysw.make_HT(np.eye(3), P_CD)    

# T_BC_target_before_open_gripper = T_BM1 @ np.linalg.inv(T_CD)
# T_BC_target_after_open_gripper  = T_BM2 @ np.linalg.inv(T_CD)
# print(f"Before open gripper: \n{T_BC_target_before_open_gripper}\n")
# print(f"After open gripper: \n{T_BC_target_after_open_gripper}\n")
# print(f"Moving Arm: {moved_arm}")


Before open gripper: 
[[ 0.      0.     -1.      0.6559]
 [-1.      0.     -0.      0.2289]
 [ 0.      1.      0.     -0.0643]
 [ 0.      0.      0.      1.    ]]

After open gripper: 
[[ 0.      0.     -1.      0.7159]
 [-1.      0.     -0.      0.2289]
 [ 0.      1.      0.     -0.0643]
 [ 0.      0.      0.      1.    ]]

Moving Arm: left


In [11]:
# {A}: vision frame
# {B}: "link_head_2" frame
# {S}: "link_torso_5" frame 
# {M}: Goal frame (object) 
# {M1}: Goal frame (object) before open the gripper 
# {M2}: Goal frame (object) after open the gripper 
# {N}: Goal frame (collecting box) 
# {C}: ee_finger_l1/r1 frame
# {D}: final fingertip frame -> same orientation of {C}

leftmost_obj_pos    = leftmost_obj_avg_sample["center_3d_avg"]
leftmost_obj_3Dsize = leftmost_obj_avg_sample["bbox_3d_size_avg"]

P_BM = ysw.convertPos_vision_to_linkhead2(leftmost_obj_pos)

fk_result = ysw.get_FK(robot, "link_torso_5", "link_head_2")
T_SB = fk_result["T"]
R_SB = fk_result["rotation"]
P_SB = fk_result["position"]

P_BM_extended = np.append(P_BM, 1.0)
P_SM = T_SB @ P_BM_extended
P_SM = P_SM[:3]

moved_arm = None
if (P_BM[1] <= 0):     
    moved_arm = "right"
else:     
    moved_arm = "left"

if (moved_arm == "right"):
    R1 = ysw.rotMat_z(90.0*ysw.DEG2RAD)      # "link_torso_5" frame 기준으로 z축 +90도 회전
    R2 = ysw.rotMat_x(-90.0*ysw.DEG2RAD)     # After R1, converted "link_torso_5" frame 기준으로 x축 -90도 회전
    R_SC_desired = R1 @ R2                   # Make Grasping orientation same as D435i frame orientation angle (it does not mean that x,y,z axis directions are same)  
elif (moved_arm == "left"):
    R1 = ysw.rotMat_x(90.0*ysw.DEG2RAD)      # "link_torso_5" frame 기준으로 x축 +90도 회전
    R2 = ysw.rotMat_y(-90.0*ysw.DEG2RAD)     # After R1, converted "link_torso_5" frame 기준으로 y축 -90도 회전
    R_SC_desired = R1 @ R2                   # Make Grasping orientation same as D435i frame orientation angle (it does not mean that x,y,z axis directions are same) 

P_SM1 = P_SM + np.array([-abs(ysw.GRIPPER_OFFSET_BEFORE_GRASP), 0.0, 0.0])      # x-offset before open the gripper
R_SM1 = R_SC_desired
T_SM1 = ysw.make_HT(R_SM1, P_SM1)    

P_SM2 = P_SM
R_SM2 = R_SC_desired
T_SM2 = ysw.make_HT(R_SM2, P_SM2)    

# "ee_finger_l1/r1" to final end-effector -> should be physically measured
pos_from_eeFinger_to_fingertip = np.array([0, 0, ysw.GRIPPER_Z_OFFSET_FROM_LINK_EE_FINGER]) # P_CD
P_CD = pos_from_eeFinger_to_fingertip
T_CD = ysw.make_HT(np.eye(3), P_CD)    

T_SC_target_before_open_gripper = T_SM1 @ np.linalg.inv(T_CD)
T_SC_target_after_open_gripper  = T_SM2 @ np.linalg.inv(T_CD)
print(f"Before open gripper: \n{T_SC_target_before_open_gripper}\n")
print(f"After open gripper: \n{T_SC_target_after_open_gripper}\n")
print(f"Moving Arm: {moved_arm}")


*************** [FK of 'link_head_2' from 'link_torso_5' Starts] ***************

[Result]

T:
[[ 0.5736  0.      0.8192  0.022 ]
 [ 0.      1.      0.      0.    ]
 [-0.8192  0.      0.5736  0.2001]
 [ 0.      0.      0.      1.    ]]


position [m]: [0.022  0.     0.2001]
rpy [rad]: [0.     0.9599 0.    ]
rpy [deg]: [ 0. 55.  0.]

**************** [FK of 'link_head_2' from 'link_torso_5' Ends] ****************



Before open gripper: 
[[ 0.      0.     -1.      0.4218]
 [-1.      0.     -0.      0.0897]
 [ 0.      1.      0.     -0.3051]
 [ 0.      0.      0.      1.    ]]

After open gripper: 
[[ 0.      0.     -1.      0.4518]
 [-1.      0.     -0.      0.0897]
 [ 0.      1.      0.     -0.3051]
 [ 0.      0.      0.      1.    ]]

Moving Arm: left


<span style="
    color:#B8860B;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:50px;
    letter-spacing:1px;
    line-height:1.2;
">
Move to Leftmost object on Shelf (before open gripper)
</span>

In [ ]:
pos_before_open_gripper = T_SC_target_before_open_gripper[:3,3]
ori_before_open_gripper = ysw.rotMat_to_rpy(T_SC_target_before_open_gripper[:3,:3])       

if (moved_arm == "right"):      # Object is in right side of robot
    cartesian_position_targets_sample = [    
                                            {
                                                "reference_link": "link_torso_5",
                                                "target_link": "ee_finger_r1",
                                                "pos": pos_before_open_gripper,                  
                                                "rpy": ori_before_open_gripper,                    
                                            },                 
                                        ]
    
elif (moved_arm == "left"):       # Object is in left side of robot
    cartesian_position_targets_sample = [    
                                            {
                                                "reference_link": "link_torso_5",
                                                "target_link": "ee_finger_l1",
                                                "pos": pos_before_open_gripper,                  
                                                "rpy": ori_before_open_gripper,                    
                                            },                 
                                        ]


# [rv = "function"] form is Necessary when "get_return=False"
rv = ysw.positionCtrl_cartesian_wholeBody(
    robot, 
    position_targets=cartesian_position_targets_sample, 
    joint_targets=None,
    control_hold_time=0.5,
    get_return=True
)

*************** [Cartesian Position Control(Whole-Body) Starts] ****************

[INFO] Add (Whole-Body) Cartesian target 0: 'link_head_2'->'ee_finger_l1', pos=[ 0.6559  0.2289 -0.0643], rpy=[ 1.5708 -0.     -1.5708]

**************** [Cartesian Position Control(Whole-Body) Ends] *****************





<span style="
    color:#B8860B;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:50px;
    letter-spacing:1px;
    line-height:1.2;
">
Open Gripper before Grasp
</span>

In [10]:
object_meanDim = np.mean(leftmost_obj_3Dsize)       # mean value of 3D size of object to grasp
desiredGripperPos = object_meanDim 

gripper = ysw.Gripper()
gripper.connect(device_name=ysw.GRIPPER_DEVICE_NAME, protocol_version=ysw.DXL_PROTOCOL_VERSION, baudrate=ysw.DXL_BAUDRATE)
gripper.test_ping()


# desiredGripperPos_mm = desiredGripperPos * 1000.0 + ysw.GRIPPER_SAFETY_OPEN_OFFSET
desiredGripperPos_mm = desiredGripperPos * 1000.0
if desiredGripperPos_mm >= ysw.GRIPPER_POS_RANGE_LINEAR:
    desiredGripperPos_mm = ysw.GRIPPER_POS_RANGE_LINEAR - 3.0

if (moved_arm == "right"):      # Object is in right side of robot
    gripper.initialize_setting(dxl_id=ysw.GRIPPER_ID_R, operation_mode=ysw.DXL_CURRENT_BASED_POSITION_CONTROL_MODE)
    gripper.current_based_position_control_linear(ysw.GRIPPER_ID_R, current_ampere_mag=0.02, linear_pos=desiredGripperPos_mm)
    time.sleep(1)
    gripper.disable(ysw.GRIPPER_ID_R)
elif (moved_arm == "left"):       # Object is in left side of robot
    gripper.initialize_setting(dxl_id=ysw.GRIPPER_ID_L, operation_mode=ysw.DXL_CURRENT_BASED_POSITION_CONTROL_MODE)
    gripper.current_based_position_control_linear(ysw.GRIPPER_ID_L, current_ampere_mag=0.02, linear_pos=desiredGripperPos_mm)
    time.sleep(1)
    gripper.disable(ysw.GRIPPER_ID_L)

************************* [Gripper Connection Starts] **************************

[INFO] SUCCESS: open the port: /dev/rby1_gripper
[INFO] SUCCESS: change the baudrate 2000000

************************* [Gripper Connection is Done] *************************



************************** [Gripper Ping Test Starts] **************************

[INFO] [DXL ID:0] SUCCESS: Dynamixel model number is 1030
[INFO] [DXL ID:1] SUCCESS: Dynamixel model number is 1030

************************* [Gripper Ping Test is Done] **************************



********************* [Gripper [DXL ID:1] setting Starts] **********************

[INFO] SUCCESS: [DXL ID:1] Torque is Disabled
[INFO] SUCCESS: [DXL ID:1] Set Current-based Position Control Mode
[INFO] SUCCESS: [DXL ID:1] Torque is Enabled

********************* [Gripper [DXL ID:1] setting is Done] *********************



********** [Gripper [DXL ID:1] Current-based Position Control Starts] **********

Desired Linear Position = 88.0mm
[INFO] [DXL ID:1]

<span style="
    color:#B8860B;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:50px;
    letter-spacing:1px;
    line-height:1.2;
">
Move to Leftmost object on Shelf (after open gripper)
</span>

In [11]:
pos_after_open_gripper = T_BC_target_after_open_gripper[:3,3]
ori_after_open_gripper = ysw.rotMat_to_rpy(T_BC_target_after_open_gripper[:3,:3])       

if (moved_arm == "right"):      # Object is in right side of robot
    cartesian_position_targets_sample = [    
                                            {
                                                "reference_link": "link_head_2",
                                                "target_link": "ee_finger_r1",
                                                "pos": pos_after_open_gripper,                  
                                                "rpy": ori_after_open_gripper,                    
                                            },                 
                                        ]
    
elif (moved_arm == "left"):       # Object is in left side of robot
    cartesian_position_targets_sample = [    
                                            {
                                                "reference_link": "link_head_2",
                                                "target_link": "ee_finger_l1",
                                                "pos": pos_after_open_gripper,                  
                                                "rpy": ori_after_open_gripper,                    
                                            },                 
                                        ]


# [rv = "function"] form is Necessary when "get_return=False"
rv = ysw.positionCtrl_cartesian_wholeBody(
    robot, 
    position_targets=cartesian_position_targets_sample, 
    joint_targets=None,
    control_hold_time=0.5,
    get_return=False
)

*************** [Cartesian Position Control(Whole-Body) Starts] ****************

[INFO] Add (Whole-Body) Cartesian target 0: 'link_head_2'->'ee_finger_l1', pos=[ 0.7159  0.2289 -0.0643], rpy=[ 1.5708 -0.     -1.5708]

**************** [Cartesian Position Control(Whole-Body) Ends] *****************





<span style="
    color:#B8860B;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:50px;
    letter-spacing:1px;
    line-height:1.2;
">
Close Gripper to Grasp
</span>

In [12]:
object_minDim = min(leftmost_obj_3Dsize)       # minimum value of 3D size of object to grasp
desiredGripperPos = object_minDim
 
grasp_current = 0.2     # It can be changed

gripper = ysw.Gripper()
gripper.connect(device_name=ysw.GRIPPER_DEVICE_NAME, protocol_version=ysw.DXL_PROTOCOL_VERSION, baudrate=ysw.DXL_BAUDRATE)
gripper.test_ping()

if (moved_arm=="right"):      # Object is in right side of robot
    gripper.initialize_setting(dxl_id=ysw.GRIPPER_ID_R, operation_mode=ysw.DXL_CURRENT_CONTROL_MODE)
    gripper.enable(ysw.GRIPPER_ID_R)
    gripper.write_current_ampere(ysw.GRIPPER_ID_L, grasp_current)
elif (moved_arm=="left"):       # Object is in left side of robot
    gripper.initialize_setting(dxl_id=ysw.GRIPPER_ID_L, operation_mode=ysw.DXL_CURRENT_CONTROL_MODE)
    gripper.enable(ysw.GRIPPER_ID_L)
    gripper.write_current_ampere(ysw.GRIPPER_ID_L, grasp_current)

************************* [Gripper Connection Starts] **************************

[INFO] SUCCESS: open the port: /dev/rby1_gripper
[INFO] SUCCESS: change the baudrate 2000000

************************* [Gripper Connection is Done] *************************



************************** [Gripper Ping Test Starts] **************************

[INFO] [DXL ID:0] SUCCESS: Dynamixel model number is 1030
[INFO] [DXL ID:1] SUCCESS: Dynamixel model number is 1030

************************* [Gripper Ping Test is Done] **************************



********************* [Gripper [DXL ID:1] setting Starts] **********************

[INFO] SUCCESS: [DXL ID:1] Torque is Disabled
[INFO] SUCCESS: [DXL ID:1] Set Current Control Mode
[INFO] SUCCESS: [DXL ID:1] Torque is Enabled

********************* [Gripper [DXL ID:1] setting is Done] *********************



********************** [Gripper [DXL ID:1] Enable Starts] **********************

[INFO] SUCCESS: [DXL ID:1] Torque is Enabled

*******************

<span style="
    color:#B8860B;
    font-family:'Segoe UI', 'Arial', sans-serif;
    font-weight:700;
    font-size:50px;
    letter-spacing:1px;
    line-height:1.2;
">
Move to Table
</span>

In [13]:
desired_pos = ysw.DEMO_AFTER_GRASP_POS_260723 
ysw.positionCtrl_joint_22dof(robot, desired_pos, minimum_time=5.0, control_hold_time=0.5)

******************** [22-DoF Joint Position Control Starts] ********************


********************* [22-DoF Joint Position Control Ends] *********************





True

In [11]:
desired_pos = ysw.DEMO_BASIC_POS_260723 
ysw.positionCtrl_joint_22dof(robot, desired_pos, minimum_time=5.0, control_hold_time=0.5)

******************** [22-DoF Joint Position Control Starts] ********************


********************* [22-DoF Joint Position Control Ends] *********************





True

In [ ]:
desired_pos = ysw.DEMO_VISION_BASIC_POS_260723 
ysw.positionCtrl_joint_22dof(robot, desired_pos, minimum_time=5.0, control_hold_time=0.5)

<span style="color:gray; font-weight:bold; font-size:50px;">
Get Forward Kinematics
</span>

In [17]:
# [Link]:     "link_right_arm_6" 
# -> [Joint]: "tool_right"
# -> [Link]:  "ee_right" 
# -> [Joint]: "gripper_finger_r1"
# -> [Link]:  "ee_finger_r1"    (r1이 ee_right 기준으로 +x 방향, 즉 정면에서 볼때 가까운 방향의 그리퍼임)

# ysw.get_FK(robot, reference_link="base", target_link="link_torso_5")
# ysw.get_FK(robot, reference_link="base", target_link="ee_right")
# ysw.get_FK(robot, reference_link="link_head_2", target_link="ee_right")
# ysw.get_FK(robot, reference_link="link_head_2", target_link="ee_finger_r1")


# End-effector Position Measurement
# fk_result = ysw.get_FK(robot, reference_link="link_head_2", target_link="link_right_arm_1") 
fk_result = ysw.get_FK(robot, reference_link="link_head_2", target_link="link_right_arm_5")    # Actual(pitch joint):             0.65 / FK: 0.652    
fk_result = ysw.get_FK(robot, reference_link="link_head_2", target_link="link_right_arm_6")    # Actual(pitch joint):             0.65 / FK: 0.652 
fk_result = ysw.get_FK(robot, reference_link="link_head_2", target_link="ee_right")            # Actual(gripper prismatic joint): 0.84 / FK: 0.7781 / Not important
fk_result = ysw.get_FK(robot, reference_link="link_head_2", target_link="ee_finger_r1")        # Actual(gripper fingertip):       0.9  / FK: 0.8531 /
# "ee_finger_r1" is located at starting point of prsimatic joint of gripper

************* [FK of 'link_right_arm_5' from 'link_head_2' Starts] *************

[Result]

T:
[[ 1.     -0.006   0.0013 -0.0209]
 [ 0.006   1.     -0.0067 -0.2164]
 [-0.0013  0.0067  1.     -0.6522]
 [ 0.      0.      0.      1.    ]]


position [m]: [-0.0209 -0.2164 -0.6522]
rpy [rad]: [0.0067 0.0013 0.006 ]
rpy [deg]: [0.3856 0.0734 0.342 ]

************** [FK of 'link_right_arm_5' from 'link_head_2' Ends] **************



************* [FK of 'link_right_arm_6' from 'link_head_2' Starts] *************

[Result]

T:
[[ 0.9988  0.0489  0.0013 -0.0209]
 [-0.0488  0.9988 -0.0067 -0.2164]
 [-0.0016  0.0067  1.     -0.6522]
 [ 0.      0.      0.      1.    ]]


position [m]: [-0.0209 -0.2164 -0.6522]
rpy [rad]: [ 0.0067  0.0016 -0.0489]
rpy [deg]: [ 0.381   0.0945 -2.8   ]

************** [FK of 'link_right_arm_6' from 'link_head_2' Ends] **************



***************** [FK of 'ee_right' from 'link_head_2' Starts] *****************

[Result]

T:
[[ 0.9988  0.0489  0.0013 -0.0211]
 [

<span style="color:gray; font-weight:bold; font-size:50px;">
[Test] 3D information of colored object
</span>

In [11]:
d435i = ysw.D435i(rgb_on=True, depth_on=True, pixel=(640,480), fps=30)
d435i.initialize()

d435i.startPipeline()

try:
    while True:
        color_frame, depth_frame = d435i.getRawFrames()

        if color_frame is None or depth_frame is None:
            continue

        color_image, depth_image = d435i.frameToImage(color_frame, depth_frame)
        depth_colorMap = d435i.convertDepthToColorMap(depth_image, alpha=0.03)

        detected_objects, color_image, mask = d435i.detectColorObjectContour3D(
            color_image=color_image,
            depth_frame=depth_frame,
            mask_list=ysw.HSV_MASK_BLACK,       # Can be changed
            minimum_area=1000,                   # Can be changed]
            maximum_depth=0.8                   # ignore distant object larger than x[m] 
        )

        for obj in detected_objects:
            print(
                f"[object {obj['id']}] "
                f"center_3d={np.round(obj['center_3d'], 3)}, "
                f"bbox_3d_size={np.round(obj['bbox_3d_size'], 3)}"
            )

        d435i.imshow("color_image", color_image)
        d435i.imshow("depth_image", depth_colorMap)
        d435i.imshow("mask", mask)

        key = d435i.waitKey(1)
        if key == 27:
            break

finally:
    d435i.stopPipeline()
    d435i.destroyAllWindows()

[INFO] D435i pipeline started
[object 0] center_3d=[-0.03   0.166  0.647], bbox_3d_size=[0.03  0.115 0.066]
[object 1] center_3d=[-0.256  0.14   0.64 ], bbox_3d_size=[0.038 0.155 0.113]
[object 2] center_3d=[-0.146  0.134  0.644], bbox_3d_size=[0.039 0.162 0.091]
[object 0] center_3d=[-0.03   0.165  0.644], bbox_3d_size=[0.029 0.115 0.262]
[object 1] center_3d=[-0.258  0.147  0.642], bbox_3d_size=[0.11  0.156 0.298]
[object 2] center_3d=[-0.146  0.142  0.637], bbox_3d_size=[0.061 0.161 0.3  ]
[object 0] center_3d=[-0.03   0.165  0.654], bbox_3d_size=[0.03  0.113 0.269]
[object 1] center_3d=[-0.258  0.146  0.64 ], bbox_3d_size=[0.118 0.158 0.3  ]
[object 2] center_3d=[-0.146  0.14   0.641], bbox_3d_size=[0.061 0.162 0.302]
[object 0] center_3d=[-0.031  0.17   0.65 ], bbox_3d_size=[0.03  0.112 0.277]
[object 1] center_3d=[-0.258  0.15   0.638], bbox_3d_size=[0.137 0.144 0.316]
[object 2] center_3d=[-0.147  0.145  0.645], bbox_3d_size=[0.064 0.164 0.317]
[object 0] center_3d=[-0.03   0.17

<span style="color:#B8860B; font-weight:bold; font-size:50px;">
Move object left to right hand
</span>

In [13]:
### Move to Pre-posture (Joint Position Control)
pre_posture = ysw.DEMO3_POS     # ZERO_POS, READY_POS, PACK_POS, DEMO_POS, DEMO3_POS
ysw.positionCtrl_joint_22dof(robot, pre_posture, minimum_time=5.0, control_hold_time=0.5)

******************** [22-DoF Joint Position Control Starts] ********************


********************* [22-DoF Joint Position Control Ends] *********************





True

<span style="color:#B8860B; font-weight:bold; font-size:50px;">
Close Right gripper to grasp
</span>

In [ ]:
object_minDim = min(second_avg["bbox_3d_size_avg"])
desiredGripperPos = object_minDim 

desiredGripperPos_mm = desiredGripperPos * 1000.0 + ysw.GRIPPER_SAFETY_CLOSE_OFFSET
if desiredGripperPos_mm >= ysw.GRIPPER_POS_RANGE_LINEAR:
    desiredGripperPos_mm = ysw.GRIPPER_POS_RANGE_LINEAR

gripper.enable(ysw.GRIPPER_ID_R)
gripper.current_based_position_control_linear(ysw.GRIPPER_ID_R, current_ampere_mag=0.2, linear_pos=desiredGripperPos_mm)

<span style="color:#B8860B; font-weight:bold; font-size:50px;">
Move to Large Box
</span>

In [ ]:
box_pos = largest_avg["center_3d_avg"]
box_3Dsize = largest_avg["bbox_3d_size_avg"]
box_maxDepth = max(object_3Dsize)




            
desiredPos_linkhead2_eefinger_before_open_gripper, desiredPos_linkhead2_eefinger_after_open_gripper = ysw.calculateEEpos_forGrasp(object_pos, object_3Dsize)

if (desiredPos_linkhead2_eefinger_before_open_gripper[1] <= 0):      # Object is in right side of robot
    fk_result = ysw.get_FK(robot, reference_link="link_head_2", target_link="ee_finger_r1")
    pos_original = fk_result["position"]
    rpy_original = fk_result["rpy"]
     
    R1 = ysw.rotMat_z(90.0*ysw.DEG2RAD)      # "link_head_2" frame 기준으로 z축 +90도 회전
    R2 = ysw.rotMat_x(-90.0*ysw.DEG2RAD)     # After R1, converted "link_head_2" frame 기준으로 x축 -90도 회전
    R = R1 @ R2
    rpy_changed = ysw.rotMat_to_rpy(R)
    
    ### Cartesian Position Control example (Whole-Body version)
    cartesian_position_targets_sample = [    
                                            {
                                                "reference_link": "link_head_2",
                                                "target_link": "ee_finger_r1",
                                                "pos": desiredPos_linkhead2_eefinger_before_open_gripper,                  
                                                "rpy": rpy_changed,                    
                                            },                 
                                        ]
    
else:       # Object is in left side of robot
    fk_result = ysw.get_FK(robot, reference_link="link_head_2", target_link="ee_finger_l1")    
    pos_original = fk_result["position"]
    rpy_original = fk_result["rpy"]
    
    R1 = ysw.rotMat_x(90.0*ysw.DEG2RAD)    # "link_head_2" frame 기준으로 x축 +90도 회전
    R2 = ysw.rotMat_y(-90.0*ysw.DEG2RAD)    # "link_head_2" frame 기준으로 y축 -90도 회전
    R = R1 @ R2
    rpy_changed = ysw.rotMat_to_rpy(R)

    ### Cartesian Position Control example (Whole-Body version)
    cartesian_position_targets_sample = [    
                                            {
                                                "reference_link": "link_head_2",
                                                "target_link": "ee_finger_l1",
                                                "pos": desiredPos_linkhead2_eefinger_before_open_gripper,                  
                                                "rpy": rpy_changed,                    
                                            },                 
                                        ]


# [rv = "function"] form is Necessary when "get_return=False"
rv = ysw.positionCtrl_cartesian_wholeBody(
    robot, 
    position_targets=cartesian_position_targets_sample, 
    # joint_targets=joint_targets_sample, 
    joint_targets=None,
    # control_hold_time=0.5,
    get_return=True
)

<span style="color:gray; font-weight:bold; font-size:50px;">
[Test] Move
</span>

In [ ]:
moved_arm = "left"

T_SC_target_before_open_gripper = np.array([
[ 0.  ,    0. ,    -1. ,     0.4518],
 [-1. ,     0. ,    -0. ,     0.0897],
 [ 0.  ,    1. ,     0.  ,   -0.3051],
 [ 0.   ,   0.  ,    0.   ,   1.    ]]
, dtype=float)

pos_before_open_gripper = T_SC_target_before_open_gripper[:3,3]
ori_before_open_gripper = ysw.rotMat_to_rpy(T_SC_target_before_open_gripper[:3,:3])

if (moved_arm == "right"):      # Object is in right side of robot
    cartesian_position_targets_sample = [    
                                            {
                                                "reference_link": "link_torso_5",
                                                "target_link": "ee_finger_r1",
                                                "pos": pos_before_open_gripper,                  
                                                "rpy": ori_before_open_gripper,                    
                                            },                 
                                        ]
elif (moved_arm == "left"):       # Object is in left side of robot
    cartesian_position_targets_sample = [    
                                            {
                                                "reference_link": "link_torso_5",
                                                "target_link": "ee_finger_l1",
                                                "pos": pos_before_open_gripper,                  
                                                "rpy": ori_before_open_gripper,                    
                                            },                 
                                        ]

rv = ysw.positionCtrl_cartesian_wholeBody(
    robot, 
    position_targets=cartesian_position_targets_sample, 
    joint_targets=None,
    control_hold_time=0.5,
    get_return=True
)